In [ ]:
import numpy as np
import random

### This notebook generates the data and plots the variance in outcomes(cooperation rate,average payoff,etc.) with different payoff parameters

In [ ]:
# NOTATION: Q is the MUTANT strategy, P is the resident strategy

# We expect vectors of size 5 in here, last element is the probability of initial cooperation

# A1, A2,... each represents a row of the transition matrix

# We should have a 5x5 matrix here for the 5 states CC,CD,DC,DD,A 

# P = [pcc,pcd,pdc,pdd,p0] where p0 is the probability of initial cooperation

def payoff_calculation(P,Q,b,q):
    c = 1 
    R_ = b-c                                        # Reward R
    S_ = -c                                         # Sucker's payoff S
    T_ = b                                          # Temptation T
    P_ = 0                                          # mutual defection punishment P
    Q_ = q                                          # Absorbing state payoff
    A1 = np.array([P[0]*Q[0]*(P_trans_CC),P[0]*(1-Q[0])*(P_trans_CC),(1-P[0])*Q[0]*(P_trans_CC),(1-P[0])*(1-Q[0])*P_trans_CC,1-P_trans_CC])
    A2 = np.array([P[1]*Q[2]*P_trans_CD,P[1]*(1-Q[2])*P_trans_CD,(1-P[1])*Q[2]*P_trans_CD,(1-P[1])*(1-Q[2])*P_trans_CD,1-P_trans_CD])
    A3 = np.array([P_trans_CD*P[2]*Q[1],P_trans_CD*P[2]*(1-Q[1]),P_trans_CD*(1-P[2])*Q[1],P_trans_CD*(1-P[2])*(1-Q[1]),1-P_trans_CD])
    A4 = np.array([P_trans_DD*P[3]*Q[3],P_trans_DD*P[3]*(1-Q[3]),P_trans_DD*(1-P[3])*Q[3],P_trans_DD*(1-P[3])*(1-Q[3]),1-P_trans_DD])
    A5 = np.array([0,0,0,0,1])
    matrix = np.array([A1,A2,A3,A4,A5])
    v0 = [P[4]*Q[4],P[4]*(1-Q[4]),(1-P[4])*Q[4],(1-P[4])*(1-Q[4]),0]
    V= (1-delta)*np.dot(v0,np.linalg.inv((np.eye(5)-delta*matrix)))  #Stationary state vector/ mean distribution                                                #Normalizing
    pA = np.dot(np.array([R_,S_,T_,P_,Q_]),V)                        #Payoff of first player
    pB = np.dot(np.array([R_,T_,S_,P_,Q_]),V)
    return list([pA,pB,V])                                           #returns the mean distribution and the payoffs of the players

In [ ]:
delta = 0.9   # Continuation probability / payoff discounting parameter
P_trans_CC=1                                              #  Probability of being in state 1 in next round when in CC
P_trans_CD = 0.8                                          # "                                                     " CD
P_trans_DD = 0.5                                      # "                                                     " DD
init_strat = [0.01,0.01,0.01,0.01,0.01]                   # initial strategy the population will be full off 

In [ ]:
# Calculates the net payoff in a population of N with j mutants
# Notations P: resident, Q: mutant
# Notations PP: payoff of P against P, PQ: payoff of P against Q, and so on

def net_payoff(PP,PQ,QP,QQ,N,j):
    Pnet = ((N-j-1)*PP + j*PQ)/(N-1)
    Qnet = ((j-1)*QQ + (N-j)*QP)/(N-1)
    return [Pnet,Qnet]

In [ ]:
# beta is the selection strength

def fixation_prob(P,Q,N,beta,b,q):
    P_case = payoff_calculation(P,P,b,q)
    P_vec = P_case[2]                       # mean distributuion when P plays against P
    PP = P_case[0]                          #Payoff of P against itself
    mixed= payoff_calculation(P,Q,b,q)
    PQ = mixed[0]                           #Payoff of P against Q
    QP = mixed[1]                           #Payoff of Q against P
    Q_case = payoff_calculation(Q,Q,b,q)    
    Q_vec = Q_case[2]                       # mean distribution when Q vs Q
    QQ = Q_case[0] 
    
    #Calculating the fixation prob using the formula
    S = 1
    factor = 1
    for i in range(1,N):
        pi = net_payoff(PP,PQ,QP,QQ,N,i)
        alpha = np.exp(-beta*(pi[1]-pi[0]))
        factor*=alpha
        S+=factor
    fixprob = 1/S
    stats = [PP,QQ,P_vec,Q_vec]
    return fixprob,stats    



In [ ]:
# Calculates time spent in state-1 from mean distribution
def t_state1(vector):
    return 1-vector[4]

# Calculates cooperation rate from mean distribution
def c_rate(vector):
    return (vector[0]+(vector[1]+vector[2])/2)/(1-vector[4])


#Checks if a strategy is partner 
def partner_check(arr,b,q,ep=0.1):
    c = 1 
    R_ = b-c                                        # Reward R
    S_ = -c                                         # Sucker's payoff S
    T_ = b                                          # Temptation T
    P_ = 0                                          # mutual defection punishment P
    Q_ = q                                          
    z2 = P_trans_CD
    z4 = P_trans_DD
    p1 = arr[0]
    p2 = arr[1]
    p3 = arr[2]
    p4 = arr[3]
    p0 = arr[4]
    cond0 = p0-(1-ep)                # p_{0} should be in neighbourhood of 1 or p0>(1-ep) 
    cond1 = p1 -(1-ep)               # p_{1} should be in neighbourhood of 1 or p1>(1-ep) 
    flag = 0
    #Next conditions are from Akin's lemma
    cond2 = (T_ - R_)*(1 - delta)*(1 - (1 - p3)*delta*z2) + (R_ - Q_)*(1 - z2)*(delta*z2*(p2 - p3) - 1) - (R_ - S_)*(1 - delta)*(1 - p2)*delta*z2 
    cond3 = T_ - R_ + delta * (Q_ - T_ + (-1 + delta) * P_ * (-1 + p2) * z2 - Q_ * z2 + delta * Q_ * z2 - delta * p2 * Q_ * z2 + p2 * R_ * z2 + (-1 + p4) * (-R_ + delta * (Q_ - T_) + T_) * z4 + delta * (p2 - p4) * (Q_ - R_) * z2 * z4)
    if cond0>0 and cond1>0 and cond2<0 and cond3<0:
        flag=1        
    return flag
    

# Main function
# Takes mutants in every iteration and checks if it can replace the resident strategy



#parameters: popN is population size, time is number of generations, Beta is selection strength
def selection(q,benefit, popN = 100,time = 500000,Beta = 1):
    #Initial paramters for the run 
    b = benefit 
    c = 1 
    R_ = b-c                                        # Reward R
    S_ = -c                                         # Sucker's payoff S
    T_ = b                                          # Temptation T
    P_ = 0                                          # mutual defection punishment P
    Q_ = q                                          # Absorbing state payoff
    init_state = payoff_calculation(init_strat,init_strat,b,Q_)      
    Avg_payoff = init_state[0]   
    coop_rate = c_rate(init_state[2])
    T_state1 = t_state1(init_state[2])
    partners = 0
    #Above code initiates a population of ALL-D with a random mutant to check against for the first time
    temp_C = coop_rate
    temp_s1 = T_state1
    temp_part = partners
    temp_payoff = Avg_payoff


    strat = []                                      # Always consists of two elements, [resident,mutant] for each iteration 
    strat.append(init_strat)                        # initial resident population of random population
    mutant = np.random.uniform(0, 1, 5)             # Drawing five values from 0 to 1 for initial mutant
    strat.append(mutant)                           

    
    for i in range(time):
        prob,pi = fixation_prob(b = b ,q =  Q_, P = strat[0],Q = strat[1],N=popN,beta = Beta)  #Should give the fixation probability of strat Q in resident P population
        
        #Scenario when mutant gets fixed
        if random.random()<prob:
            del strat[0]                          #Eliminate resident strategy
            # Update the payoff, cooperation rate etc.
            temp_payoff = pi[1]
            temp_C = c_rate(pi[3])
            temp_s1 = t_state1(pi[3])
            temp_part = partner_check(strat[0],b,q)
        else:
        #Mutant doesn't get selected 
            del strat[1]                          #Eliminate previous mutant
        Avg_payoff+=temp_payoff
        coop_rate+=temp_C
        T_state1+=temp_s1
        partners+=temp_part
        mutant = np.random.uniform(0, 1, 5)   #New mutant introduced
        strat.append(mutant)
        
    return np.array([coop_rate/(time+1),Avg_payoff/(time+1),T_state1/(time+1), partners/(time+1)])

### In the following block we vary the payoff parameters and record the results 

In [ ]:
Q = [-3,-2,-1,0,1,2,3]    # Degraded state payoffs
Bs = [0.5,1,1.1,1.5,2,3.5,5,7]   # B/c ratios (c =1) 
pop = 100
time = 500000
strength = 1
def averager(i,j,trials = 10):
    Results = np.zeros(4)
    for k in range(trials):
        Results+= selection(i,j)
    return Results/trials
    
heatmap  = [[averager(i,j) for j in Bs] for i in Q]

In [ ]:
heatmap = np.array(heatmap,dtype=object)
np.save("Qvspayoff.npy", heatmap)

### Plotting the results

In [ ]:
import numpy as np

heatmap = np.load("Qvspayoff.npy",allow_pickle=True)

In [ ]:
Q = [-3,-2,-1,0,1,2,3]
Bs = [0.5,1,1.1,1.5,2,3.5,5,7]

L1 = len(Q)
L2 = len(Bs)

coop_rate = np.zeros((L1,L2))
payoffs = np.zeros((L1,L2))
partners  = np.zeros((L1,L2))
time1 =  np.zeros((L1,L2))
for i in range(L1):
    for j in range(L2):
        coop_rate[i,j] = heatmap[i,j][0]
        payoffs[i,j] = heatmap[i,j][1]
        time1[i,j] = heatmap[i,j][2]
        partners[i,j] = heatmap[i,j][3]        

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import string  # for 'a', 'b', 'c', 'd'
import matplotlib as mpl
# Example placeholders:
# coop_rate, payoffs, time1, partners, Bs, Q = ...

mpl.rcParams.update(mpl.rcParamsDefault)  # reset all rcParams to default


fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)

heatmaps = [
    (coop_rate, "Cooperation rate"),
    (payoffs, "Mean Payoff"),
    (time1, "Time spent in healthy state"),
    (partners, "Partner probability"),
]

x_labels = Bs
y_labels = Q
x_ticks = np.arange(len(x_labels))
y_ticks = np.arange(len(y_labels))

for idx, (ax, (matrix, cbar_label), label) in enumerate(zip(axes.flat, heatmaps, string.ascii_lowercase)):
    im = ax.imshow(matrix, cmap="viridis", origin='lower')

    # Each subplot gets its own colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label(cbar_label, rotation=270, labelpad=15, fontsize=11)
    cbar.outline.set_visible(False)

    # Axis ticks
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

    # Hide redundant tick labels
    if idx < 2:      # top row
        ax.tick_params(labelbottom=False)
    if idx % 2 == 1: # right column
        ax.tick_params(labelleft=False)

    # Bold subplot label (a, b, c, d)
    ax.text(
        -0.12, 1.05, label, transform=ax.transAxes,
        fontsize=12, fontweight='bold', va='bottom', ha='right'
    )

# Adjust layout
plt.tight_layout(pad=1.0)

# Common axis labels (non-bold)
fig.text(0.5, 0.01, "$b/c$ ratio", ha='center', va='center', fontsize=12)
fig.text(-0.01, 0.5, "Degraded state payoff ($Q$)", ha='center', va='center',
         rotation='vertical', fontsize=12)

# Export
plt.savefig("Q_vs_benefit.pdf", bbox_inches='tight', pad_inches=0.1)
plt.show()
